# Device1

In [1]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda:0")  

In [16]:

import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18

from sklearn.model_selection import KFold

current_dir = Path.cwd() #ipynb相当于cmd环境
root_dir = current_dir.parent 
text_path = root_dir / 'data' / 'raw' / 'tabular_data.xlsx'
pkl_path =  root_dir / 'data' / 'interim' 
image_root = root_dir / 'data'  / 'raw'/ 'image' 
resnet_weight = root_dir / 'weights' / 'resnet18.pth'
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# device = torch.device("cuda:0")  
# weights = torch.tensor([1.0, 2.6132]).to(device)
# ========== 配置 ==========
CONFIG = {
	'excel_path': text_path,          # 包含 patient_id + feature_cols + aux_cols
	 'pkl_path': pkl_path / 'clinical.pkl', 
	'image_root': image_root,               # 病人影像根目录
	'save_dir':root_dir/ 'notebooks'/'result'/'Resnet',
  	'checkpoint_dir': root_dir/ 'notebooks'/'result'/'Resnet',
	'img_size': 224,
	'batch_size': 8,                            # 医学影像显存占用大，建议8-16
	'epochs': 100,
	'lr_backbone': 1e-5,                       # 影像backbone小学习率微调
	'lr_head': 1e-3,                           # 临床塔/分类头大学习率
	'weight_decay': 1e-4,
	'dropout': 0.4,
	'patience': 15,
	'use_aux_task': True,                      # 是否用 aux_label 做多任务辅助
	'aux_weight': 0.3,                         # 辅助损失权重
	'pos_weight': 3,                         # III型(正类)加权
	'seed': 42,
	'device': torch.device("cuda:0") ,
	'resnet_weight':resnet_weight,
  'threshold': 0.25,   
  'num_folds': 5,
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)

# 数据集

## AOJ Image DataSet and Dataloader

In [3]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms  
from pathlib import Path
class AOJDataset(Dataset):
	def __init__(self, image_dir, patient_ids,transform=None):
		self.root_dir = Path(image_dir)
		self.transform = transform
		self.samples = []

		for p_id in patient_ids:
			# 1st改：仅仅遍历Fold中的图片
			patient_folder = self.root_dir / p_id
			if not patient_folder.is_dir():
				continue
			# for patient_folder in self.root_dir.iterdir():
				# 1st: 遍历所有病人
				# if patient_folder.is_dir():
			patient_id = patient_folder.name
			# 2nd: 遍历左右关节
			for side_folder in patient_folder.iterdir():
				if side_folder.is_dir():
					folder_name = side_folder.name

					side = 'left' if '左' in folder_name else 'right'
					label = -1
					# 三型关节为正类 1， 其余型为负类 0
					if '3型' in folder_name or 'III型' in folder_name: label = 1
					else: label = 0
					
					if label == -1: print(f'非正常标签出现,病人编号:{patient_id}'); continue

					# 3rd: 遍历关节的全部影像
					for img_file in side_folder.glob('*.*'):
						if img_file.suffix.lower() in ['.png', '.jpg', '.jpeg', '.tif']:
							self.samples.append({
								'path': str(img_file),
								'label': label,
								'patient_id': patient_id,
								'side': side
							})
		print(f"总计发现病人：{len(set(s['patient_id'] for s in self.samples))}位")
		print(f"总计加载影像样本：{len(self.samples)}张")

	def __len__(self):
		return len(self.samples)
	
	def __getitem__(self, idx):
		sample = self.samples[idx]
		img = Image.open(sample['path']).convert('L') #转成灰度模式减少噪声干燥

		if self.transform:
			img = self.transform(img)

		label = sample['label']
		p_id = sample['patient_id']
		side = sample['side']
		joint_id = f"{p_id}_{side}"
		return img, torch.tensor(label, dtype= torch.long) , joint_id


Tranform


class Compose:
	def __init__(self, transforms):
		self.transforms = transforms

	def __call__(self, img):
		for t in self.transforms:
			img = t(img)          # 顺序执行
		return img
 <font color = '#006400'>
 compose 是一个可调用对象，实现了__call__方法，把多个transform组成一个新的可调用对象
 </font>

## 预处理

In [4]:
train_transforms = transforms.Compose([
	transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
	transforms.ColorJitter(brightness=0.3, contrast=0.3), # 强化骨骼边缘对比度
	transforms.Grayscale(num_output_channels=3), 
		transforms.RandomRotation(5), # 模拟拍摄摆位误差
	transforms.RandomAffine(degrees=0, translate=(0.02, 0.02)),
	transforms.ToTensor(),
		transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [5]:
import torchvision.transforms as transforms
val_transforms = transforms.Compose([
	transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
	transforms.Grayscale(num_output_channels=3),
	transforms.ToTensor(), # 转化为 Tensor (会将像素值归一化到 [0.0, 1.0])
	transforms.Normalize(
		mean=[0.485, 0.456, 0.406], 
		std=[0.229, 0.224, 0.225]
	)	# 使用 ImageNet 的经典参数
])

# 模型

## ResNet-18 Base on RadImageNet

In [6]:
import torch
import torch.nn as nn
import torchvision.models as models

class AOJBIResNET(nn.Module):
	def __init__(self, pretrained_path = None):
		super(AOJBIResNET, self).__init__()
		self.resnet = models.resnet18(weights=None)

		num_ftrs= self.resnet.fc.in_features
		self.resnet.fc = nn.Sequential(
						nn.Linear(num_ftrs, 2),
		)

		if pretrained_path:
			self.load_smart_weights(pretrained_path)


	def load_smart_weights(self, path):
		print(f"正在加载权重文件: {path}")
		state_dict = torch.load(path, map_location='cpu')
		# 自动拆包 (针对 'state_dict' 等外壳)
		possible_keys = ['state_dict', 'model', 'net', 'model_state_dict']
		for key in possible_keys:
			if key in state_dict and isinstance(state_dict[key], dict):
				print(f"发现权重被包裹在 '{key}' 中，正在拆包...")
				state_dict = state_dict[key]
				break
		model_state_dict = self.state_dict()
		new_state_dict = {}
		matched_layers = 0
		print(f"开始匹配 (文件包含 {len(state_dict)} 个键)...")
		for k, v in state_dict.items():
			# 多重前缀清洗
			key_clean = k.replace('_orig_mod.', '')
			key_clean = key_clean.replace('module.', '')
			target_key = "resnet." + key_clean
			# 开始尝试匹配
			final_key = None
			if target_key in model_state_dict:
				final_key = target_key
			elif key_clean in model_state_dict:
				final_key = key_clean
			if final_key:
				if v.shape == model_state_dict[final_key].shape:
					new_state_dict[final_key] = v
					matched_layers += 1
				else:
					# 形状不匹配跳过 
					pass
		self.load_state_dict(new_state_dict, strict=False)
		print(f" 权重加载完成！成功匹配: {matched_layers}")

		if matched_layers == 0:
				print(f" 警告：匹配数为 0！")
				print(f"   文件里的第一条Key (清洗后): {key_clean}")
				print(f"   模型里的第一条Key (预期): {list(model_state_dict.keys())[0]}")
				raise RuntimeError("无法匹配任何权重，请检查上述 Key 的差异。")

	def load_local_weights(self, path):
		state_dict = torch.load(path, map_location='cpu')

		_ = self.load_state_dict(state_dict, strict=False)
		# print(f"本地权重加载结果:{msg}")

	def forward(self, x):
		return self.resnet(x)

# 评价函数

In [7]:
from sklearn.model_selection import KFold
import numpy as np

## Joint-Level EVA

In [8]:
from collections import defaultdict
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, average_precision_score
def evaluate_joint_level(model, dataloader, device):
	model.eval()
	joint_probs = defaultdict(list)
	joint_labels = {}

	with torch.no_grad():
		for images, labels, j_ids in dataloader:
			images = images.to(device)
			logits = model(images)
			# probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
					# ========== 兼容两种输出格式 ==========
			if logits.dim() == 1 or logits.size(-1) == 1:
				# 单输出 [B] 或 [B,1] → sigmoid
				probs = torch.sigmoid(logits).cpu().numpy()
				if probs.ndim == 2:
					probs = probs.squeeze(-1)
			else:
				# 双输出 [B, 2] → softmax 取第1类（III型）
				probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
			for i in range(len(j_ids)):
				jid = j_ids[i]
				joint_probs[jid].append(probs[i])
				joint_labels[jid] = labels[i].item()

	# 执行关节级聚合
	all_labels, final_preds, all_probs = [], [], []
	final_joint_ids = []
	for jid in joint_probs:
		avg_prob = np.mean(joint_probs[jid]) # 聚合该关节的所有切片
		pred = 1 if avg_prob > 0.25 else 0
		all_probs.append(avg_prob)
		# final_preds.append(pred)
		all_labels.append(joint_labels[jid])
		final_joint_ids.append(jid)
	
	all_labels = np.array(all_labels)
	all_probs = np.array(all_probs)
	final_preds = (all_probs >= 0.25).astype(int)
	# 检测错误关节
	fp_joints = []   # 假阳性：预测1，实际0
	fn_joints = []   # 假阴性：预测0，实际1
	for idx, jid in enumerate(sorted(joint_probs.keys())):
		pred = final_preds[idx]
		label = all_labels[idx]
		prob = all_probs[idx]
		
		if pred == 1 and label == 0:
			fp_joints.append(jid
				# {
				# 'joint_id': jid,
				# 'prob': round(float(prob), 4),
				# 'threshold': round(float(threshold), 4)
			# }
			)
		elif pred == 0 and label == 1:
			fn_joints.append(jid
				# {
				# 'joint_id': jid,
				# 'prob': round(float(prob), 4),
				# 'threshold': round(float(threshold), 4)
			# }
			)
	print(f"误诊为III型的关节: {fp_joints}")
	print(f"漏诊的III型关节: {fn_joints}")







	# ========== 指标计算（与多模态版本完全一致）==========
	acc = accuracy_score(all_labels, final_preds)
	prec = precision_score(all_labels, final_preds, zero_division=0)
	rec = recall_score(all_labels, final_preds, zero_division=0)
	f1 = f1_score(all_labels, final_preds, zero_division=0)

	try:
		auc = roc_auc_score(all_labels, all_probs)
	except ValueError:
		auc = 0.5
	try:
		ap = average_precision_score(all_labels, all_probs)
	except ValueError:
		ap = 0.0

	try:
		tn, fp, fn, tp = confusion_matrix(all_labels, final_preds).ravel()
		sens = tp / (tp + fn) if (tp + fn) > 0 else 0
		spec = tn / (tn + fp) if (tn + fp) > 0 else 0
		ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
		npv = tn / (tn + fn) if (tn + fn) > 0 else 0
	except ValueError:
		tn = fp = fn = tp = 0
		sens = spec = ppv = npv = 0

	metrics = {
		'auc': float(auc),
		'ap': float(ap),
		'threshold': float(0.25),
		'accuracy': float(acc),
		'precision': float(prec),
		'recall': float(rec),
		'f1': float(f1),
		'sensitivity': float(sens),
		'specificity': float(spec),
		'ppv': float(ppv),
		'npv': float(npv),
		'tp': int(tp),
		'fp': int(fp),
		'tn': int(tn),
		'fn': int(fn),
		'num_images': len(all_labels),  # 纯影像版用 num_images
		'num_positive': int(all_labels.sum()),
	}

	# 返回值与多模态版本完全一致
	return metrics, all_labels.tolist(), final_preds.tolist(), all_probs.tolist(), final_joint_ids
	# acc = accuracy_score(final_labels, final_preds)
	# auc = roc_auc_score(final_labels, final_probs)
	# f1 = f1_score(final_labels, final_preds)
	# precision = precision_score(final_labels, final_preds, zero_division=0)
	# recall = recall_score(final_labels, final_preds, zero_division=0)
	# auprc = average_precision_score(final_labels, final_probs)

	# return {
	# 			'Accuracy': acc, 
	# 			'AUC': auc, 
	# 			'F1-score': f1, 
	# 			'Precision': precision, 
	# 			'Recall': recall,
	# 			'AUPRC': auprc
	# 	}

# 5-FOLD

## RESNET TRAINING

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import argparse
import pandas as pd
import json
torch.serialization.add_safe_globals([argparse.Namespace])

from tqdm.auto import tqdm
data_root = Path(CONFIG['image_root'])
all_patient_ids =  [d.name for d in data_root.iterdir() if d.is_dir]
all_patient_ids = np.array(all_patient_ids)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
save_dir = CONFIG['save_dir']
save_dir.mkdir(parents=True, exist_ok=True)
print(f'结果保存至{save_dir}')
# 存储每一折的结果
fold_results = []
for fold, (train_idx, val_idx) in enumerate(kf.split(all_patient_ids)):
	print(f"开始训练第 {fold+1} 折...")
	
	train_patients = all_patient_ids[train_idx]
	val_patients = all_patient_ids[val_idx]
	# print(train_patients)
	train_ds = AOJDataset(data_root, train_patients, transform=train_transforms)
	val_ds = AOJDataset(data_root, val_patients, transform=val_transforms)

	train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,drop_last=True)
	val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)
	model = AOJBIResNET(CONFIG['resnet_weight']).to(device)

	weights = torch.tensor([1.0, CONFIG['pos_weight']]).to(device)
	# weights = torch.tensor([3.0]).to(device)
	# criterion = nn.BCEWithLogitsLoss(pos_weight=weights)
	criterion = nn.CrossEntropyLoss(weight=weights)
	backbone_params = [p for n, p in model.resnet.named_parameters() if "fc" not in n]
	head_params = model.resnet.fc.parameters()
	

	optimizer = torch.optim.AdamW([
		{'params': backbone_params, 'lr': CONFIG['lr_backbone']}, # 所有卷积层都在这
		{'params': head_params, 'lr':  CONFIG['lr_head']}      # 只有分类头在这
	], weight_decay=CONFIG['weight_decay'])
	scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
			optimizer, mode='max', factor=0.5, patience=10, verbose=True
		)
	'''Each Epoch'''
	history = {
		'train_loss': [],
		'val_acc': [],
	'val_auc': [],
			'Pr':[],
		'Recall':[],
		'F1':[],
		'AUPRC':[],
		'metrics_per_fold': []
	} #新增历史追踪器

	best_val_acc = 0.0
	beat_epoch = 0
	no_improve = 0
	pbar_epoch = tqdm(range(2), desc=f"Fold {fold+1}")
	for epoch in pbar_epoch: # 小样本不需要太多 epoch
		model.train()
		train_loss = 0.0
		train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{100}", leave=False)
		'''Each Batch'''
		for images, labels,_ in train_bar:
			images, labels = images.to(device), labels.to(device)
			# print(f'debug:{images.shape}')
			# print(f'debug:{labels.shape}')
			optimizer.zero_grad()
			outputs = model(images)
			# print(f'debug:{outputs.shape}')
			loss = criterion(outputs, labels)
			loss.backward()
			optimizer.step()
			train_loss += loss.item()
			train_bar.set_postfix(loss=f"{loss.item():.4f}")
		

		# metrics = evaluate_patient_level(model, val_loader, device)
		# metrics = evaluate(model, val_loader, device)
		metrics,_,_,_,_ = evaluate_joint_level(model, val_loader, device)
		val_acc = metrics['accuracy']
		val_auc = metrics['auc']
		val_pr =metrics['precision']
		val_recall = metrics['recall']
		val_f1 =metrics['f1']
		val_auprc = metrics['ap']

		scheduler.step(val_auc)
		# print(f"第{fold+1}fold的准确率：{val_acc}")
		# print(f"Validation Accuracy: {metrics['Accuracy']:.4f}")
		# print(f"AUC: {metrics['AUC']:.4f}")
		# print(f"Sensitivity (识别III型能力): {metrics['Sensitivity']:.4f}")
		# print(f"Specificity (排除非III型能力): {metrics['Specificity']:.4f}")
				# 打印混淆矩阵
		# tn, fp, fn, tp = metrics['Confusion_Matrix']
		# print(f"\n混淆矩阵:\n[TN: {tn}, FP: {fp}]\n[FN: {fn}, TP: {tp}]")
		if val_auc > best_val_acc:
			best_val_acc = val_auc
			beat_epoch = epoch
			torch.save(model.state_dict(), save_dir / f'best_model_fold{fold+1}.pth')
		else:
			no_improve +=1
		
		# 这样信息会显示在进度条末尾，而不会新开一行
		pbar_epoch.set_postfix({
				'Current_Auc': f"{val_auc:.4f}",
				'Best_Auc': f"{best_val_acc:.4f}"
		})
		avg_train_loss = train_loss / len(train_loader)
		history['train_loss'].append(avg_train_loss)
		history['val_acc'].append(val_acc)
		history['val_auc'].append(val_auc)
		history['Pr'].append(val_pr)
		history['Recall'].append(val_recall)
		history['F1'].append(val_f1)
		history['AUPRC'].append(val_auprc)
		if no_improve >= 15:
			print(f"\n早停于 Epoch {epoch} (Best: Epoch {beat_epoch}, AUC={best_val_acc:.3f})")
			break
		# pbar_epoch.set_postfix(Val_Acc=f"{val_acc:.4f}", Best=f"{best_val_acc:.4f}")
	df_history = pd.DataFrame({
		'epoch': range(1, len(history['train_loss']) + 1),
		'loss': history['train_loss'],
		'acc': history['val_acc'],
		'auc': history['val_auc'],
		'pr': history['Pr'],
		'recall': history['Recall'],
		'f1': history['F1'],
		'auprc': history['AUPRC']

	})
	df_history.to_csv(save_dir / f'fold_{fold+1}_log.csv', index=False)
	fold_results.append(best_val_acc)

print(f"\n5-Fold 平均AUC: {sum(fold_results)/5:.2f}")
# 损失函数，roc，混淆矩阵，热力图

# 统计

##  resnet 

In [14]:
#!/usr/bin/env python3
"""
evaluate_densenet_5fold.py
读取 DenseNet 训练保存的 5-fold 权重，重新评估并输出汇总
"""

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import KFold
from sklearn.metrics import (
	accuracy_score, precision_score, recall_score, f1_score,
	roc_auc_score, confusion_matrix, average_precision_score,
	roc_curve
)
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict
from PIL import Image
import os
import pickle
# ==================== 配置（必须与训练时一致） ====================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_root = CONFIG['image_root']
save_dir = CONFIG['checkpoint_dir']
save_dir.mkdir(parents=True, exist_ok=True)


def main():
	print("=" * 60)
	print("DenseNet 5-Fold 重新评估")
	print("=" * 60)
	
	# 重建与训练时完全相同的病人列表和划分
	all_patient_ids = [d.name for d in data_root.iterdir() if d.is_dir()]
	all_patient_ids = np.array(all_patient_ids)
	print(f"总病人: {len(all_patient_ids)}")
	
	kf = KFold(n_splits=5, shuffle=True, random_state=42)
	
	all_fold_metrics = []
	fold_results = {}      # 用于绘图
	all_labels_total = []
	all_preds_total = []
	
	for fold, (train_idx, val_idx) in enumerate(kf.split(all_patient_ids)):
		print(f"\n--- Fold {fold+1} ---")
		
		val_patients = all_patient_ids[val_idx]
		ckpt_path = save_dir / f'best_model_fold{fold+1}.pth'
		
		if not ckpt_path.exists():
			print(f"  权重不存在: {ckpt_path}，跳过")
			continue
		
		# 验证集
		val_ds = AOJDataset(data_root, val_patients, transform=val_transforms)
		val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4)
		
		# 加载模型
		model = AOJBIResNET()
		model.load_state_dict(torch.load(ckpt_path, map_location=device))
		model = model.to(device)
		model.eval()
		metrics, labels,  preds, probs, joint_ids  = evaluate_joint_level(
			model, val_loader, device#, False,0.25
		)
		# metrics, labels,  preds, probs = evaluate_patient_level(
		# 	model, val_loader, CONFIG['device'],CONFIG['threshold']
		# )
		# best_thresh = metrics['threshold']
		# print(best_thresh)
		
		all_fold_metrics.append(metrics)
		fold_results[fold] = (labels, probs, joint_ids)
		# fold_results.append((labels, probs))
		# all_labels_total.extend(labels)
		# all_preds_total.extend(preds)

		print(f"  AUC: {metrics['auc']:.3f} | AP: {metrics['ap']:.3f}")
		print(f"  Acc: {metrics['accuracy']:.3f} | Sen: {metrics['sensitivity']:.3f} | "
				f"Spe: {metrics['specificity']:.3f} | Pr:{metrics['precision']:.3f}")
		print(f"  PPV: {metrics['ppv']:.3f} | NPV: {metrics['npv']:.3f} | "
				f"F1: {metrics['f1']:.3f}")
		print(f"  TP={metrics['tp']} FP={metrics['fp']} TN={metrics['tn']} FN={metrics['fn']}")

	# ========== 汇总表格 ==========
	print("\n" + "=" * 60)
	print(f"5-Fold 汇总 (Threshold=0.25)")
	print("=" * 60)
	
	keys = ['auc', 'ap', 'accuracy', 'sensitivity', 'specificity', 'precision','ppv', 'npv', 'f1']
	print(f"{'Metric':<<15} " + " ".join([f"F{i:<7}" for i in range(1, 6)]) + " Mean±Std")
	for k in keys:
		vals = [m[k] for m in all_fold_metrics]
		mean = np.mean(vals)
		std = np.std(vals)
		print(f"{k:<15} " + " ".join([f"{v:.3f}  " for v in vals]) + f" {mean:.3f}±{std:.3f}")
	results = {
		'threshold': 0.25,	
		'fold_metrics': all_fold_metrics,
		'fold_results': fold_results,
		'mean': {k: float(np.mean([m[k] for m in all_fold_metrics])) for k in keys},
		'std': {k: float(np.std([m[k] for m in all_fold_metrics])) for k in keys}
	}
	# 保存结果
	with open(CONFIG['checkpoint_dir'] / '5fold_results.pkl', 'wb') as f:
		pickle.dump(results, f)
	print("\n结果已保存到 5fold_results.pkl")




		# 评估
		# metrics = evaluate_joint_level(
		#     model, val_loader, device
		# )

		
	#     all_fold_metrics.append(metrics)
	#     # # fold_results[fold] = (labels, probs)
	#     # all_labels_total.extend(labels)
	#     # all_preds_total.extend(preds)
		
	#     print(f"  AUC: {metrics['AUC']:.3f} | AP: {metrics['AUPRC']:.3f}")
	#     print(f"  Acc: {metrics['Accuracy']:.3f} | Sen: {metrics['Recall']:.3f} ")
	#     print(f"  Pr: {metrics['Precision']:.3f} | Rec: {metrics['Recall']:.3f} | "
	#           f"F1: {metrics['F1-score']:.3f}")

	
	# if len(all_fold_metrics) == 0:
	#     print("没有成功评估任何 fold！")
	#     return
	
	# # ========== 汇总表格 ==========
	# print("\n" + "=" * 70)
	# print("5-Fold 汇总结果")
	# print("=" * 70)
	
	# keys = ['AUC', 'AUPRC', 'Accuracy', 'Recall', 
	#         'Precision', 'F1-score']
	
	# print(f"{'Metric':<15} " + " ".join([f"F{i:<7}" for i in range(1, 6)]) + " Mean±Std")
	# print("-" * 85)
	
	# for k in keys:
	#     vals = [m[k] for m in all_fold_metrics]
	#     mean = np.mean(vals)
	#     std = np.std(vals)
	#     print(f"{k:<15} " + " ".join([f"{v:.3f}  " for v in vals]) + f" {mean:.3f}±{std:.3f}")
	
	# # ========== 绘图 ==========

	# # ========== 保存 CSV ==========
	# df = pd.DataFrame(all_fold_metrics)
	# df.index = [f'Fold{i+1}' for i in range(len(df))]
	# df.loc['Mean'] = df.mean()
	# df.loc['Std'] = df.std()
	# csv_path = save_dir / '5fold_densenet_results.csv'
	# df.to_csv(csv_path)
	# print(f"\n结果已保存: {csv_path}")
	# print("=" * 70)
	with open(CONFIG['checkpoint_dir']/ 'roc_results.pkl', 'wb') as f:
		pickle.dump(fold_results, f)
	print("\n结果已保存到 roc_results.pkl")


if __name__ == '__main__':
	main()

DenseNet 5-Fold 重新评估
总病人: 215

--- Fold 1 ---
总计发现病人：43位
总计加载影像样本：612张


/tmp/ipykernel_430136/936128797.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=device))


误诊为III型的关节: ['105_left', '116_left', '116_right', '119_left', '123_left', '123_right', '198_left', '198_right', '239_left', '239_right']
漏诊的III型关节: ['130_left', '49_left', '88_right', '90_right']
  AUC: 0.948 | AP: 0.918
  Acc: 0.837 | Sen: 0.875 | Spe: 0.815 | Pr:0.737
  PPV: 0.737 | NPV: 0.917 | F1: 0.800
  TP=28 FP=10 TN=44 FN=4

--- Fold 2 ---
总计发现病人：43位
总计加载影像样本：664张


/tmp/ipykernel_430136/936128797.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=device))


误诊为III型的关节: ['137_right', '143_left', '145_left', '153_left', '153_right', '215_left', '23_left', '23_right', '33_left', '44_left', '44_right', '5_left', '7_left', '92_right']
漏诊的III型关节: ['145_right', '243_left', '96_left']
  AUC: 0.903 | AP: 0.829
  Acc: 0.802 | Sen: 0.880 | Spe: 0.770 | Pr:0.611
  PPV: 0.611 | NPV: 0.940 | F1: 0.721
  TP=22 FP=14 TN=47 FN=3

--- Fold 3 ---
总计发现病人：43位
总计加载影像样本：536张


...

误诊为III型的关节: ['120_left', '120_right', '128_left', '128_right', '166_left', '166_right', '241_left', '241_right', '245_left', '25_left', '25_right', '67_left', '75_left']
漏诊的III型关节: ['121_right', '35_right', '66_right', '67_right', '94_left']
  AUC: 0.909 | AP: 0.882
  Acc: 0.791 | Sen: 0.808 | Spe: 0.783 | Pr:0.618
  PPV: 0.618 | NPV: 0.904 | F1: 0.700
  TP=21 FP=13 TN=47 FN=5

--- Fold 4 ---
总计发现病人：43位
总计加载影像样本：699张


/tmp/ipykernel_430136/936128797.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=device))


误诊为III型的关节: ['103_left', '165_left', '165_right', '19_right', '202_left', '204_left', '210_left', '238_left', '77_left']
漏诊的III型关节: ['108_right', '46_left']
  AUC: 0.950 | AP: 0.930
  Acc: 0.872 | Sen: 0.920 | Spe: 0.852 | Pr:0.719
  PPV: 0.719 | NPV: 0.963 | F1: 0.807
  TP=23 FP=9 TN=52 FN=2

--- Fold 5 ---
总计发现病人：43位
总计加载影像样本：580张


/tmp/ipykernel_430136/936128797.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=device))


误诊为III型的关节: ['106_right', '111_left', '156_right', '16_left', '223_left', '22_left', '237_left', '237_right', '32_right', '38_left', '38_right', '55_left', '55_right', '99_right']
漏诊的III型关节: ['207_left']
  AUC: 0.956 | AP: 0.943
  Acc: 0.826 | Sen: 0.963 | Spe: 0.763 | Pr:0.650
  PPV: 0.650 | NPV: 0.978 | F1: 0.776
  TP=26 FP=14 TN=45 FN=1

5-Fold 汇总 (Threshold=0.25)
Metric<<<<<<<<< F1       F2       F3       F4       F5       Mean±Std
auc             0.948   0.903   0.909   0.950   0.956   0.933±0.022
ap              0.918   0.829   0.882   0.930   0.943   0.900±0.041
accuracy        0.837   0.802   0.791   0.872   0.826   0.826±0.028
sensitivity     0.875   0.880   0.808   0.920   0.963   0.889±0.052
specificity     0.815   0.770   0.783   0.852   0.763   0.797±0.033
precision       0.737   0.611   0.618   0.719   0.650   0.667±0.052
ppv             0.737   0.611   0.618   0.719   0.650   0.667±0.052
npv             0.917   0.940   0.904   0.963   0.978   0.940±0.028
f1              